Phase 1 - Environment Configuration & Drive Mounting

In [ ]:
from google.colab import drive
import warnings
import zipfile

warnings.filterwarnings('ignore')

print("Mounting Google Drive...")
drive.mount('/content/drive')

print("Installing required libraries...")
!pip install -q sdv pandas numpy matplotlib seaborn scikit-learn

import os
import shutil
import pandas as pd
import numpy as np
import gc
import time
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

print("\nPhase 1 Complete: Environment is ready.")

Mounting Google Drive...
Mounted at /content/drive
Installing required libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.8/204.8 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.0 MB/s eta 0:00:00

Phase 1 Complete: Environment is ready.


Phase 2 - Dataset Ingestion, Labeling & Preprocessing

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import gc
import shutil

local_final_csv = '/content/Final_Dataset_Perfect.csv'
drive_final_csv = '/content/drive/MyDrive/Final_Labeled_Dataset.csv'
base_dir = "CICIoT2023_Data/CICIoT2023_CSVs"
TARGET_SAMPLES = 100000

print("Extracting dataset...")
if not os.path.exists(base_dir):
    with zipfile.ZipFile('/content/drive/MyDrive/CICIoT2023_CSVs.zip', 'r') as zip_ref:
        zip_ref.extractall("CICIoT2023_Data")

folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]
folders.sort()

if os.path.exists(local_final_csv):
    os.remove(local_final_csv)

first_folder_path = os.path.join(base_dir, folders[0])
first_csv = [f for f in os.listdir(first_folder_path) if f.endswith('.csv')][0]
header_df = pd.read_csv(os.path.join(first_folder_path, first_csv), nrows=0)
header_df['Label'] = 'Temp'
header_df.to_csv(local_final_csv, index=False)

print(f"Processing {len(folders)} folders locally...")

for folder in folders:
    folder_path = os.path.join(base_dir, folder)
    csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

    temp_list = []
    for csv_file in csv_files:
        file_path = os.path.join(folder_path, csv_file)
        temp_df = pd.read_csv(file_path, on_bad_lines='skip', low_memory=False)
        temp_df['Label'] = folder
        temp_list.append(temp_df)

    class_df = pd.concat(temp_list, ignore_index=True)
    del temp_list

    class_df.replace([np.inf, -np.inf], 9999999, inplace=True)
    class_df.fillna(0, inplace=True)

    if len(class_df) > TARGET_SAMPLES:
        class_df = class_df.sample(n=TARGET_SAMPLES, random_state=42)
        status = f"SAMPLED to exactly {TARGET_SAMPLES:,}"
    else:
        status = f"KEPT FULL ({len(class_df):,})"

    class_df.to_csv(local_final_csv, mode='a', header=False, index=False)
    print(f"[OK] {folder:<25} {status}")

    del class_df
    gc.collect()

print("\nPhysically copying to Drive (waiting for completion)...")
shutil.copy(local_final_csv, drive_final_csv)
print("Copy complete.")

print("Verifying directly from Drive...")
verify_df = pd.read_csv(drive_final_csv)
print(f"Total rows in Drive file: {len(verify_df):,}")
under_100k = verify_df['Label'].value_counts()[verify_df['Label'].value_counts() < 100000]
print(f"Classes under 100,000: {len(under_100k)}")

Extracting dataset...
Processing 34 folders locally...
[OK] Backdoor_Malware          KEPT FULL (3,218)
[OK] Benign_Final              SAMPLED to exactly 100,000
[OK] BrowserHijacking          KEPT FULL (5,859)
[OK] CommandInjection          KEPT FULL (5,409)
[OK] DDoS-ACK_Fragmentation    SAMPLED to exactly 100,000
[OK] DDoS-HTTP_Flood           KEPT FULL (28,790)
[OK] DDoS-ICMP_Flood           SAMPLED to exactly 100,000
[OK] DDoS-ICMP_Fragmentation   SAMPLED to exactly 100,000
[OK] DDoS-PSHACK_FLOOD         SAMPLED to exactly 100,000
[OK] DDoS-RSTFINFLOOD          SAMPLED to exactly 100,000
[OK] DDoS-SYN_Flood            SAMPLED to exactly 100,000
[OK] DDoS-SlowLoris            KEPT FULL (23,426)
[OK] DDoS-SynonymousIP_Flood   SAMPLED to exactly 100,000
[OK] DDoS-TCP_Flood            SAMPLED to exactly 100,000
[OK] DDoS-UDP_Flood            SAMPLED to exactly 100,000
[OK] DDoS-UDP_Fragmentation    SAMPLED to exactly 100,000
[OK] DNS_Spoofing              SAMPLED to exactly 100,000
[O

Phase 3 - Generative Augmentation via CTGAN

In [ ]:
import joblib
joblib.parallel.DEFAULT_N_JOBS = 1
import os, gc, time, pandas as pd
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

local_final_csv = '/content/Final_Dataset_Local.csv'
local_synthetic_csv = '/content/Synthetic_100K_Local.csv'
drive_synthetic_csv = '/content/drive/MyDrive/Synthetic_100K.csv'
TARGET_COUNT = 100000

# Read from the guaranteed clean LOCAL file
print("Loading verified local dataset...")
df = pd.read_csv(local_final_csv, low_memory=False)

class_counts = df['Label'].value_counts()
remaining_classes = class_counts[class_counts < TARGET_COUNT].index.tolist()

print(f"\nTarget locked: {len(remaining_classes)} classes to augment.")

# Initialize local synthetic file
temp_header = df.head(0)
temp_header.to_csv(local_synthetic_csv, index=False)

for attack_name in remaining_classes:
    print(f"\nStarting: {attack_name}")

    real_data = df[df['Label'] == attack_name].copy()
    needed_count = TARGET_COUNT - len(real_data)
    features = real_data.drop(columns=['Label'])

    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(features)
    synthesizer = CTGANSynthesizer(metadata, epochs=100, verbose=False)

    start_time = time.time()
    synthesizer.fit(features)
    train_time = round((time.time() - start_time) / 60, 2)
    print(f"Training completed in {train_time} minutes.")

    synthetic_features = synthesizer.sample(num_rows=needed_count)
    synthetic_features['Label'] = attack_name

    # Save locally first (Safe)
    synthetic_features.to_csv(local_synthetic_csv, mode='a', header=False, index=False)

    # Copy to Drive periodically (Safe)
    os.system(f'cp "{local_synthetic_csv}" "{drive_synthetic_csv}"')

    print(f"Generated {needed_count:,} samples. Saved to Drive.")

    del real_data, features, synthesizer, synthetic_features
    gc.collect()

print("\nPhase 3 Complete: All 13 classes forced to exactly 100,000.")

Loading verified local dataset...

Target locked: 13 classes to augment.

Starting: Recon-OSScan
Training completed in 47.28 minutes.
Generated 1,741 samples. Saved to Drive.

Starting: Recon-PortScan
Training completed in 39.8 minutes.
Generated 17,716 samples. Saved to Drive.

Starting: DoS-HTTP_Flood
Training completed in 36.71 minutes.
Generated 28,139 samples. Saved to Drive.

Starting: DDoS-HTTP_Flood
Training completed in 14.63 minutes.
Generated 71,210 samples. Saved to Drive.

Starting: DDoS-SlowLoris
Training completed in 11.52 minutes.
Generated 76,574 samples. Saved to Drive.

Starting: DictionaryBruteForce
Training completed in 6.07 minutes.
Generated 86,936 samples. Saved to Drive.

Starting: BrowserHijacking
Training completed in 2.53 minutes.
Generated 94,141 samples. Saved to Drive.

Starting: CommandInjection
Training completed in 2.28 minutes.
Generated 94,591 samples. Saved to Drive.

Starting: SqlInjection
Training completed in 2.25 minutes.
Generated 94,755 sample

Phase 4: Dataset Integration & Stratified Shuffling

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.utils import shuffle

drive_real = '/content/drive/MyDrive/Final_Labeled_Dataset.csv'
drive_synthetic = '/content/drive/MyDrive/Synthetic_100K.csv'
drive_final_merged = '/content/drive/MyDrive/Final_Balanced_Trainset.csv'

# Force copy to bypass cache
os.system(f'cp "{drive_real}" "/content/real.csv"')
os.system(f'cp "{drive_synthetic}" "/content/synthetic.csv"')

# Load and merge
df_real = pd.read_csv("/content/real.csv", low_memory=False)
df_synthetic = pd.read_csv("/content/synthetic.csv", low_memory=False)

df_final = pd.concat([df_real, df_synthetic], ignore_index=True)
df_final = shuffle(df_final, random_state=42)

df_final.to_csv(drive_final_merged, index=False)
print(f"Phase 4 Complete! Final merged size: {len(df_final):,} rows.")

Phase 4 Complete! Final merged size: 3,400,000 rows.


Phase 5: Model Training & Results

In [ ]:
import pandas as pd
import numpy as np
import os, shutil, gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, QuantileTransformer, PolynomialFeatures, StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
import tensorflow.keras.backend as K

# Clean up session
K.clear_session()

# --- CRITICAL CHANGE: DISABLE MIXED PRECISION ---
# We use standard float32 for 100% stability.
# tf.keras.mixed_precision.set_global_policy('mixed_float16') <--- REMOVED

!pip install -q catboost
from catboost import CatBoostClassifier, Pool

# Automatically find the correct Drive path
if os.path.exists('/content/drive/My Drive'):
    drive_path = '/content/drive/My Drive'
elif os.path.exists('/content/drive/MyDrive'):
    drive_path = '/content/drive/MyDrive'
else:
    raise Exception("Google Drive not found. Please mount it manually via the sidebar.")

drive_final_merged = f'{drive_path}/Final_Balanced_Trainset.csv'
local_train = '/content/final_train.csv'
local_train_poly = '/content/X_train_poly.npy'
local_test_poly = '/content/X_test_poly.npy'

print("Copying dataset locally...")
shutil.copy(drive_final_merged, local_train)

print("Loading dataset...")
df = pd.read_csv(local_train, low_memory=False)

X = df.drop(columns=['Label'])
y = df['Label']

del df
gc.collect()

le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)

print("Casting to float32 for Quantile Transformer...")
X = X.astype(np.float32)

print("Applying Quantile Transformation...")
qt = QuantileTransformer(output_distribution='normal', random_state=42)
X_qt = qt.fit_transform(X)

del X
gc.collect()

print("Splitting dataset...")
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_qt, y_encoded, test_size=0.2, random_state=42)

X_train_raw_cb = X_train_raw.astype(np.float16)
X_test_raw_cb = X_test_raw.astype(np.float16)

del X_qt
gc.collect()

# --- Chunked Polynomial Feature Engineering ---
print("Generating Polynomial Features and Scaling...")
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly.fit(X_train_raw[:1000])
total_features = len(poly.get_feature_names_out())

print("Fitting StandardScaler on sample polynomial features...")
scaler = StandardScaler()
sample_poly = poly.transform(X_train_raw[:5000])
scaler.fit(sample_poly)
del sample_poly
gc.collect()

print("Processing Train chunks...")
train_chunks = np.array_split(X_train_raw, 12)
np.save(local_train_poly, np.empty((len(X_train_raw), total_features), dtype=np.float16))

train_mmap = np.load(local_train_poly, mmap_mode='r+')
start_row = 0
for i, chunk in enumerate(train_chunks):
    print(f"  Transforming and writing train chunk {i+1}/12...")
    # Generate in float32, Scale in float32, Save as float16
    poly_chunk = poly.transform(chunk)
    poly_chunk = scaler.transform(poly_chunk)
    train_mmap[start_row:start_row + len(chunk)] = poly_chunk.astype(np.float16)
    start_row += len(chunk)
    del chunk, poly_chunk
    gc.collect()

del train_mmap, train_chunks
gc.collect()

print("Processing Test chunks...")
test_chunks = np.array_split(X_test_raw, 4)
np.save(local_test_poly, np.empty((len(X_test_raw), total_features), dtype=np.float16))

test_mmap = np.load(local_test_poly, mmap_mode='r+')
start_row = 0
for i, chunk in enumerate(test_chunks):
    print(f"  Transforming and writing test chunk {i+1}/4...")
    poly_chunk = poly.transform(chunk)
    poly_chunk = scaler.transform(poly_chunk)
    test_mmap[start_row:start_row + len(chunk)] = poly_chunk.astype(np.float16)
    start_row += len(chunk)
    del chunk, poly_chunk
    gc.collect()

del test_mmap, test_chunks
del X_train_raw, X_test_raw
gc.collect()

y_train_one_hot = to_categorical(y_train, num_classes=num_classes)

print("Creating TFData Pipeline...")
X_train_poly_mmap = np.load(local_train_poly, mmap_mode='r')

# Sanity Check
if np.isnan(X_train_poly_mmap).any():
    print("WARNING: NaN detected in training features!")
else:
    print("Dataset validation passed.")

val_split_idx = int(len(X_train_poly_mmap) * 0.9)
X_train_split = X_train_poly_mmap[:val_split_idx]
y_train_split = y_train_one_hot[:val_split_idx]
X_val_split = X_train_poly_mmap[val_split_idx:]
y_val_split = y_train_one_hot[val_split_idx:]

# --- CRITICAL CHANGE: REDUCED BATCH SIZE ---
# Float32 uses more memory, so we reduce batch size from 4096 to 2048 to prevent OOM.
BATCH_SIZE = 2048

train_dataset = tf.data.Dataset.from_tensor_slices((X_train_split, y_train_split))
train_dataset = train_dataset.shuffle(buffer_size=10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val_split, y_val_split))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

del X_train_poly_mmap, X_train_split, y_train_split, X_val_split, y_val_split, y_train_one_hot
gc.collect()

print("Building PolyQuant Hybrid Architecture (STABLE MODE)...")
input_dim = total_features
inp = Input(shape=(input_dim,))

# Cross Network
x0 = inp
x_cross = x0
for _ in range(4):
    x_cross = Dense(input_dim, activation='linear')(x_cross)
    x_cross = x0 * x_cross + x_cross

# Deep Network
x_deep = Dense(512, activation='swish')(inp)
x_deep = BatchNormalization()(x_deep)
x_deep = Dropout(0.4)(x_deep)

x_deep = Dense(256, activation='swish')(x_deep)
x_deep = BatchNormalization()(x_deep)
x_deep = Dropout(0.3)(x_deep)

x_deep = Dense(128, activation='swish')(x_deep)
x_deep = BatchNormalization()(x_deep)
x_deep = Dropout(0.2)(x_deep)

combined = Concatenate()([x_cross, x_deep])
x_out = Dense(128, activation='swish')(combined)
x_out = Dropout(0.2)(x_out)
# Output layer is standard float32
output = Dense(num_classes, activation='softmax', dtype='float32')(x_out)

model = Model(inputs=inp, outputs=output)

model_filename = 'polyquant_hybrid_model.keras'
checkpoint = ModelCheckpoint(model_filename, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)

# --- OPTIMIZER SETTINGS ---
# Standard LR, but strong clipping just in case
optimizer = Adam(learning_rate=0.0001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=10, mode='max', restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

print("Training PolyQuant Hybrid Architecture...")
model.fit(
    train_dataset,
    epochs=100,
    validation_data=val_dataset,
    callbacks=[checkpoint, early_stop, reduce_lr],
    verbose=1
)

del train_dataset, val_dataset
gc.collect()

model.load_weights(model_filename)

print("Generating PolyQuant predictions...")
X_test_poly_mmap = np.load(local_test_poly, mmap_mode='r')
nn_probs = model.predict(X_test_poly_mmap, batch_size=2048)

del X_test_poly_mmap, model
gc.collect()

# Train CatBoost
print("Training CatBoost on Raw Features...")
train_pool = Pool(data=X_train_raw_cb, label=y_train)
test_pool = Pool(data=X_test_raw_cb, label=y_test)

cat_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=10,
    loss_function='MultiClass',
    task_type='GPU',
    devices='0',
    verbose=200,
    early_stopping_rounds=50
)

cat_model.fit(train_pool, eval_set=test_pool)

del X_train_raw_cb, X_test_raw_cb
gc.collect()

print("Evaluating Ensemble...")
cat_probs = cat_model.predict_proba(test_pool)

ensemble_probs = 0.4 * nn_probs + 0.6 * cat_probs
ensemble_preds = np.argmax(ensemble_probs, axis=1)

nn_preds = np.argmax(nn_probs, axis=1)
cat_preds = np.argmax(cat_probs, axis=1)

print(f"Standalone DCN-V2 Accuracy: {accuracy_score(y_test, nn_preds):.4f}")
print(f"Standalone CatBoost Accuracy: {accuracy_score(y_test, cat_preds):.4f}")
print(f"POLYQUANT ENSEMBLE ACCURACY: {accuracy_score(y_test, ensemble_preds):.4f}")

print(classification_report(y_test, ensemble_preds, target_names=le.classes_))

# --- AUTO BACKUP TO GOOGLE DRIVE ---
import shutil
from google.colab import drive
import os

print("Backing up model to Google Drive...")
drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/MyModels'
os.makedirs(drive_path, exist_ok=True)

shutil.copy(model_filename, os.path.join(drive_path, model_filename))
print("Model saved successfully to Google Drive!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
Copying dataset locally...
Loading dataset...
Casting to float32 for Quantile Transformer...
Applying Quantile Transformation...
Splitting dataset...
Generating Polynomial Features and Scaling...
Fitting StandardScaler on sample polynomial features...
Processing Train chunks...
  Transforming and writing train chunk 1/12...
  Transforming and writing train chunk 2/12...
  Transforming and writing train chunk 3/12...
  Transforming and writing train chunk 4/12...
  Transforming and writing train chunk 5/12...
  Transforming and writing train chunk 6/12...
  Transforming and writing train chunk 7/12...
  Transforming and writing train chunk 8/12...
  Transforming and writing train chunk 9/12...
  Transforming and writing train chunk 10/12...
  Transforming and writing train chunk 11/12...
  Transforming and writing train chunk 12/12...
Processing Test chunks...
  Transforming and writing test chunk 1/4...
  Tra

0:	learn: 2.8221950	test: 2.8232903	best: 2.8232903 (0)	total: 1.04s	remaining: 34m 40s
200:	learn: 0.4931249	test: 0.4958689	best: 0.4958689 (200)	total: 2m 4s	remaining: 18m 31s
400:	learn: 0.4404597	test: 0.4454928	best: 0.4454928 (400)	total: 4m 6s	remaining: 16m 21s
600:	learn: 0.4199244	test: 0.4276987	best: 0.4276987 (600)	total: 6m 8s	remaining: 14m 18s
800:	learn: 0.4077748	test: 0.4183284	best: 0.4183284 (800)	total: 8m 12s	remaining: 12m 16s
1000:	learn: 0.3991717	test: 0.4125238	best: 0.4125238 (1000)	total: 10m 16s	remaining: 10m 15s
1200:	learn: 0.3924327	test: 0.4086146	best: 0.4086146 (1200)	total: 12m 21s	remaining: 8m 13s
1400:	learn: 0.3866332	test: 0.4056897	best: 0.4056897 (1400)	total: 14m 26s	remaining: 6m 10s
1600:	learn: 0.3818349	test: 0.4035714	best: 0.4035714 (1600)	total: 16m 33s	remaining: 4m 7s
1800:	learn: 0.3772455	test: 0.4018407	best: 0.4018407 (1800)	total: 18m 41s	remaining: 2m 3s
1999:	learn: 0.3732558	test: 0.4004399	best: 0.4004399 (1999)	total: 

In [ ]:
import pandas as pd
import numpy as np
import os, shutil, gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, QuantileTransformer, PolynomialFeatures, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
import tensorflow.keras.backend as K

# --- CONFIGURATION ---
K.clear_session()
# Run in Stable Float32 mode
# tf.keras.mixed_precision.set_global_policy('mixed_float16')

!pip install -q catboost
from catboost import CatBoostClassifier, Pool

# --- PATH SETUP ---
if os.path.exists('/content/drive/My Drive'):
    drive_path = '/content/drive/My Drive'
elif os.path.exists('/content/drive/MyDrive'):
    drive_path = '/content/drive/MyDrive'
else:
    raise Exception("Google Drive not found.")

# Create a dedicated folder for this run
output_dir = f'{drive_path}/MyModels/PolyQuant_Publication_Run'
os.makedirs(output_dir, exist_ok=True)

drive_final_merged = f'{drive_path}/Final_Balanced_Trainset.csv'
local_train = '/content/final_train.csv'
local_train_poly = '/content/X_train_poly.npy'
local_test_poly = '/content/X_test_poly.npy'

# --- 1. DATA PREP ---
print("Copying dataset locally...")
shutil.copy(drive_final_merged, local_train)

print("Loading dataset...")
df = pd.read_csv(local_train, low_memory=False)
X = df.drop(columns=['Label'])
y = df['Label']
del df; gc.collect()

le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)
np.save(f'{output_dir}/classes.npy', le.classes_) # Save class names

print("Preprocessing...")
X = X.astype(np.float32)
qt = QuantileTransformer(output_distribution='normal', random_state=42)
X_qt = qt.fit_transform(X)
del X; gc.collect()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_qt, y_encoded, test_size=0.2, random_state=42)
# Save test labels for later plotting
np.save(f'{output_dir}/y_test.npy', y_test)

X_train_raw_cb = X_train_raw.astype(np.float16)
X_test_raw_cb = X_test_raw.astype(np.float16)
del X_qt; gc.collect()

# --- 2. FEATURE ENGINEERING ---
print("Generating Polynomial Features...")
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly.fit(X_train_raw[:1000])
total_features = len(poly.get_feature_names_out())

scaler = StandardScaler()
scaler.fit(poly.transform(X_train_raw[:5000]))
gc.collect()

# Process Train
train_chunks = np.array_split(X_train_raw, 12)
np.save(local_train_poly, np.empty((len(X_train_raw), total_features), dtype=np.float16))
train_mmap = np.load(local_train_poly, mmap_mode='r+')
start = 0
for chunk in train_chunks:
    train_mmap[start:start+len(chunk)] = scaler.transform(poly.transform(chunk)).astype(np.float16)
    start += len(chunk)
    del chunk; gc.collect()
del train_mmap, train_chunks; gc.collect()

# Process Test
test_chunks = np.array_split(X_test_raw, 4)
np.save(local_test_poly, np.empty((len(X_test_raw), total_features), dtype=np.float16))
test_mmap = np.load(local_test_poly, mmap_mode='r+')
start = 0
for chunk in test_chunks:
    test_mmap[start:start+len(chunk)] = scaler.transform(poly.transform(chunk)).astype(np.float16)
    start += len(chunk)
    del chunk; gc.collect()
del test_mmap, test_chunks, X_train_raw, X_test_raw; gc.collect()

# --- 3. DCN TRAINING ---
print("Training DCN-V2...")
y_train_one_hot = to_categorical(y_train, num_classes=num_classes)
X_train_poly_mmap = np.load(local_train_poly, mmap_mode='r')

val_split_idx = int(len(X_train_poly_mmap) * 0.9)
train_ds = tf.data.Dataset.from_tensor_slices((X_train_poly_mmap[:val_split_idx], y_train_one_hot[:val_split_idx]))
train_ds = train_ds.shuffle(10000).batch(2048).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_train_poly_mmap[val_split_idx:], y_train_one_hot[val_split_idx:]))
val_ds = val_ds.batch(2048).prefetch(tf.data.AUTOTUNE)

del X_train_poly_mmap, y_train_one_hot; gc.collect()

# Model Architecture
inp = Input(shape=(total_features,))
x0 = inp
x_cross = x0
for _ in range(4):
    x_cross = Dense(total_features, activation='linear')(x_cross)
    x_cross = x0 * x_cross + x_cross
x_deep = Dense(512, activation='swish')(inp)
x_deep = BatchNormalization()(x_deep)
x_deep = Dropout(0.4)(x_deep)
x_deep = Dense(256, activation='swish')(x_deep)
x_deep = BatchNormalization()(x_deep)
x_deep = Dropout(0.3)(x_deep)
combined = Concatenate()([x_cross, x_deep])
x = Dense(128, activation='swish')(combined)
x = Dropout(0.2)(x)
out = Dense(num_classes, activation='softmax', dtype='float32')(x)

model = Model(inp, out)
model_filename = f'{output_dir}/polyquant_model.keras'
checkpoint = ModelCheckpoint(model_filename, monitor='val_accuracy', save_best_only=True, mode='max', verbose=0)

optimizer = Adam(learning_rate=0.0001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=10, mode='max', restore_best_weights=True, verbose=0)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=0)

model.fit(train_ds, epochs=100, validation_data=val_ds, callbacks=[checkpoint, early_stop, reduce_lr], verbose=1)
del train_ds, val_ds; gc.collect()

model.load_weights(model_filename)
print("Generating DCN Predictions...")
X_test_poly_mmap = np.load(local_test_poly, mmap_mode='r')
nn_probs = model.predict(X_test_poly_mmap, batch_size=2048)
np.save(f'{output_dir}/nn_probs.npy', nn_probs) # SAVE PREDICTIONS
del X_test_poly_mmap, model; gc.collect()

# --- 4. CATBOOST TRAINING ---
print("Training CatBoost...")
train_pool = Pool(data=X_train_raw_cb, label=y_train)
test_pool = Pool(data=X_test_raw_cb, label=y_test)

cat_model = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=10, loss_function='MultiClass', task_type='GPU', devices='0', verbose=False, early_stopping_rounds=50)
cat_model.fit(train_pool, eval_set=test_pool)
cat_probs = cat_model.predict_proba(test_pool)
np.save(f'{output_dir}/cat_probs.npy', cat_probs) # SAVE PREDICTIONS

# --- 5. BASELINE (RANDOM FOREST) ---
print("Training Random Forest Baseline...")
rf_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42, verbose=0)
rf_model.fit(X_train_raw_cb, y_train)
rf_preds = rf_model.predict(X_test_raw_cb)
rf_acc = accuracy_score(y_test, rf_preds)

# --- 6. EVALUATION & PLOTTING ---
print("Generating Plots...")

# Load saved data back
y_test = np.load(f'{output_dir}/y_test.npy')
nn_probs = np.load(f'{output_dir}/nn_probs.npy')
cat_probs = np.load(f'{output_dir}/cat_probs.npy')
class_names = np.load(f'{output_dir}/classes.npy', allow_pickle=True)

# Calculate Final Ensemble
ensemble_probs = 0.4 * nn_probs + 0.6 * cat_probs
ensemble_preds = np.argmax(ensemble_probs, axis=1)
cat_preds = np.argmax(cat_probs, axis=1)

# Scores
acc_ens = accuracy_score(y_test, ensemble_preds)
acc_cat = accuracy_score(y_test, cat_preds)
acc_dcn = accuracy_score(y_test, np.argmax(nn_probs, axis=1))

print(f"DCN Acc: {acc_dcn:.4f}")
print(f"CatBoost Acc: {acc_cat:.4f}")
print(f"PolyQuant Ens: {acc_ens:.4f}")
print(f"Random Forest Baseline: {rf_acc:.4f}")

# Plot 1: Confusion Matrix
cm = confusion_matrix(y_test, ensemble_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(20, 18))
sns.heatmap(cm_norm, annot=False, cmap='Blues', fmt='.2f', xticklabels=class_names, yticklabels=class_names)
plt.title('Normalized Confusion Matrix: PolyQuant Ensemble', fontsize=20)
plt.ylabel('True Label', fontsize=16)
plt.xlabel('Predicted Label', fontsize=16)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f'{output_dir}/confusion_matrix.png', dpi=300)
plt.close()

# Plot 2: F1 Comparison
cat_f1 = f1_score(y_test, cat_preds, average=None)
ens_f1 = f1_score(y_test, ensemble_preds, average=None)

f1_df = pd.DataFrame({'CatBoost': cat_f1, 'PolyQuant': ens_f1}, index=class_names)
plt.figure(figsize=(12, 10))
f1_df.plot(kind='barh', width=0.8, color=['#1f77b4', '#ff7f0e'])
plt.title('Per-Class F1-Score Comparison', fontsize=16)
plt.xlabel('F1-Score', fontsize=14)
plt.xlim(0.4, 1.0)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(f'{output_dir}/f1_comparison.png', dpi=300)
plt.close()

print(f"\nALL FILES SAVED TO: {output_dir}")
print("Files ready for publication!")

Copying dataset locally...
Loading dataset...
Preprocessing...
Generating Polynomial Features...
Training DCN-V2...
Epoch 1/100
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 41s 22ms/step - accuracy: 0.6878 - loss: 1.4139 - val_accuracy: 0.7585 - val_loss: 0.7288 - learning_rate: 1.0000e-04
Epoch 2/100
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - accuracy: 0.7543 - loss: 0.7188 - val_accuracy: 0.7765 - val_loss: 0.5873 - learning_rate: 1.0000e-04
Epoch 3/100
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - accuracy: 0.7742 - loss: 0.5839 - val_accuracy: 0.7861 - val_loss: 0.5312 - learning_rate: 1.0000e-04
Epoch 4/100
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.7843 - loss: 0.5396 - val_accuracy: 0.7908 - val_loss: 0.5127 - learning_rate: 1.0000e-04
Epoch 5/100
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - accuracy: 0.7907 - loss: 0.5136 - val_accuracy: 0.7949 - val_loss: 0.5000 - learning_rate: 1.0000e-04
Epoch 6/100
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.7948 - l

Training Random Forest Baseline...


In [ ]:
import numpy as np
import pandas as pd
import os
import gc
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# --- 1. LOAD SAVED DATA FROM DRIVE ---
print("Loading predictions from Google Drive...")

# Setup Path
if os.path.exists('/content/drive/My Drive'):
    drive_path = '/content/drive/My Drive'
elif os.path.exists('/content/drive/MyDrive'):
    drive_path = '/content/drive/MyDrive'

output_dir = f'{drive_path}/MyModels/PolyQuant_Publication_Run'

# Load the predictions we saved earlier
y_test = np.load(f'{output_dir}/y_test.npy')
nn_probs = np.load(f'{output_dir}/nn_probs.npy')
cat_probs = np.load(f'{output_dir}/cat_probs.npy')

print("Data loaded successfully!")

# --- 2. CALCULATE ENSEMBLE ---
print("Calculating Ensemble Results...")
ensemble_probs = 0.4 * nn_probs + 0.6 * cat_probs
ensemble_preds = np.argmax(ensemble_probs, axis=1)
cat_preds = np.argmax(cat_probs, axis=1)
nn_preds = np.argmax(nn_probs, axis=1)

acc_ens = accuracy_score(y_test, ensemble_preds)
acc_cat = accuracy_score(y_test, cat_preds)
acc_dcn = accuracy_score(y_test, nn_preds)

print(f"DCN Accuracy:      {acc_dcn:.4f}")
print(f"CatBoost Accuracy: {acc_cat:.4f}")
print(f"Ensemble Accuracy: {acc_ens:.4f}")

# --- 3. OPTIMIZED RANDOM FOREST (Stable) ---
print("\nTraining Optimized Random Forest Baseline...")

# We need to reload the RAW features for RF.
drive_final_merged = f'{drive_path}/Final_Balanced_Trainset.csv'

print("Loading dataset for RF...")
df = pd.read_csv(drive_final_merged, low_memory=False)
X = df.drop(columns=['Label'])
y = df['Label']

# REMOVED .astype(np.float16) to prevent Infinity errors
# We will keep standard types and rely on n_jobs=1 to save memory

# We must split exactly the same way to match y_test
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split
X_train, X_test, y_train_rf, y_test_rf = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

# Clean up to save RAM before training RF
del df
gc.collect()

# Train RF with Optimized Settings
# n_estimators=50, max_depth=15, n_jobs=1
rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=15,
    n_jobs=1,       # Use 1 core to prevent RAM crash
    random_state=42,
    verbose=1
)

print("Fitting Random Forest (this may take 2-3 minutes)...")
rf.fit(X_train, y_train_rf)

rf_preds = rf.predict(X_test)
rf_acc = accuracy_score(y_test_rf, rf_preds)

print(f"\n--- FINAL COMPARISON ---")
print(f"1. Random Forest Baseline:  {rf_acc:.4f}")
print(f"2. CatBoost:               {acc_cat:.4f}")
print(f"3. PolyQuant Ensemble:     {acc_ens:.4f}")

if acc_ens > rf_acc:
    print("\nSUCCESS: PolyQuant outperforms the baseline!")
else:
    print("\nNOTE: Baseline is close. Highlight Precision/Recall in paper.")

print("\nDetailed Random Forest Report:")
print(classification_report(y_test_rf, rf_preds))

Loading predictions from Google Drive...
Data loaded successfully!
Calculating Ensemble Results...
DCN Accuracy:      0.8115
CatBoost Accuracy: 0.8288
Ensemble Accuracy: 0.8259

Training Optimized Random Forest Baseline...
Loading dataset for RF...
Fitting Random Forest (this may take 2-3 minutes)...


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:  8.7min
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed:  8.9min finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    8.9s
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed:    9.1s finished



--- FINAL COMPARISON ---
1. Random Forest Baseline:  0.7948
2. CatBoost:               0.8288
3. PolyQuant Ensemble:     0.8259

SUCCESS: PolyQuant outperforms the baseline!

Detailed Random Forest Report:
              precision    recall  f1-score   support

           0       0.98      0.94      0.96     20398
           1       0.46      0.68      0.55     19975
           2       0.88      0.91      0.89     20105
           3       0.94      0.86      0.90     19933
           4       1.00      0.98      0.99     20000
           5       0.97      0.90      0.93     19769
           6       1.00      1.00      1.00     20171
           7       1.00      0.98      0.99     19940
           8       1.00      1.00      1.00     20122
           9       1.00      1.00      1.00     19774
          10       0.53      0.20      0.29     19931
          11       0.91      0.99      0.95     20150
          12       0.45      0.65      0.53     20092
          13       0.72      0.44   

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score
import pandas as pd

# 1. Find the correct Drive Path
if os.path.exists('/content/drive/My Drive'):
    drive_path = '/content/drive/My Drive'
else:
    drive_path = '/content/drive/MyDrive'

print(f"Using Drive Path: {drive_path}")

# 2. Search for the saved predictions
search_path = f'{drive_path}/MyModels'
found = False

for root, dirs, files in os.walk(search_path):
    if "nn_probs.npy" in files:
        output_dir = root
        print(f"Found data in: {output_dir}")
        found = True
        break

if not found:
    print("ERROR: Could not find nn_probs.npy in MyModels folder.")
else:
    # 3. Load Data
    y_test = np.load(f'{output_dir}/y_test.npy')
    nn_probs = np.load(f'{output_dir}/nn_probs.npy')
    cat_probs = np.load(f'{output_dir}/cat_probs.npy')

    # Calculate predictions
    ensemble_probs = 0.4 * nn_probs + 0.6 * cat_probs
    ensemble_preds = np.argmax(ensemble_probs, axis=1)
    cat_preds = np.argmax(cat_probs, axis=1)

    # Load Class Names
    classes_path = os.path.join(output_dir, 'classes.npy')
    if os.path.exists(classes_path):
        class_names = np.load(classes_path, allow_pickle=True)
    else:
        class_names = [f"Class_{i}" for i in range(34)] # Fallback

    # 4. Generate Confusion Matrix
    cm = confusion_matrix(y_test, ensemble_preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(20, 18))
    sns.heatmap(cm_norm, annot=False, cmap='Blues', fmt='.2f', xticklabels=class_names, yticklabels=class_names)
    plt.title('Normalized Confusion Matrix: PolyQuant Ensemble', fontsize=20)
    plt.ylabel('True Label', fontsize=16)
    plt.xlabel('Predicted Label', fontsize=16)
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()

    # Save to the SAME folder where data was found
    cm_path = os.path.join(output_dir, 'confusion_matrix.png')
    plt.savefig(cm_path, dpi=300)
    plt.close()
    print(f"Saved Confusion Matrix to: {cm_path}")

    # 5. Generate F1 Comparison
    cat_f1 = f1_score(y_test, cat_preds, average=None)
    ens_f1 = f1_score(y_test, ensemble_preds, average=None)

    f1_df = pd.DataFrame({'CatBoost': cat_f1, 'PolyQuant': ens_f1}, index=class_names)
    plt.figure(figsize=(12, 10))
    f1_df.plot(kind='barh', width=0.8, color=['#1f77b4', '#ff7f0e'])
    plt.title('Per-Class F1-Score Comparison', fontsize=16)
    plt.xlabel('F1-Score', fontsize=14)
    plt.xlim(0.4, 1.0)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()

    f1_path = os.path.join(output_dir, 'f1_comparison.png')
    plt.savefig(f1_path, dpi=300)
    plt.close()
    print(f"Saved F1 Comparison to: {f1_path}")

    print("\nDONE! Check the folder path printed above.")

Using Drive Path: /content/drive/My Drive
Found data in: /content/drive/My Drive/MyModels/PolyQuant_Publication_Run
Saved Confusion Matrix to: /content/drive/My Drive/MyModels/PolyQuant_Publication_Run/confusion_matrix.png
Saved F1 Comparison to: /content/drive/My Drive/MyModels/PolyQuant_Publication_Run/f1_comparison.png

DONE! Check the folder path printed above.


<Figure size 1200x1000 with 0 Axes>

In [ ]:
import pandas as pd
import numpy as np
import os, shutil, gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, QuantileTransformer, PolynomialFeatures, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
import tensorflow.keras.backend as K

!pip install -q catboost
from catboost import CatBoostClassifier, Pool

# --- CONFIGURATION ---
K.clear_session()

# Path Setup
if os.path.exists('/content/drive/My Drive'):
    drive_path = '/content/drive/My Drive'
elif os.path.exists('/content/drive/MyDrive'):
    drive_path = '/content/drive/MyDrive'

output_dir = f'{drive_path}/MyModels/PolyQuant_Final_Run'
os.makedirs(output_dir, exist_ok=True)

drive_final_merged = f'{drive_path}/Final_Balanced_Trainset.csv'
local_train = '/content/final_train.csv'
local_train_poly = '/content/X_train_poly.npy'
local_test_poly = '/content/X_test_poly.npy'

# --- 1. DATA PREP ---
print("Copying dataset...")
shutil.copy(drive_final_merged, local_train)

print("Loading dataset...")
df = pd.read_csv(local_train, low_memory=False)
X = df.drop(columns=['Label'])
y = df['Label']
del df; gc.collect()

le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)
np.save(f'{output_dir}/classes.npy', le.classes_)

print("Preprocessing...")
X = X.astype(np.float32)
qt = QuantileTransformer(output_distribution='normal', random_state=42)
X_qt = qt.fit_transform(X)
del X; gc.collect()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_qt, y_encoded, test_size=0.2, random_state=42)
np.save(f'{output_dir}/y_test.npy', y_test)

X_train_raw_cb = X_train_raw.astype(np.float16)
X_test_raw_cb = X_test_raw.astype(np.float16)
del X_qt; gc.collect()

# --- 2. FEATURE ENGINEERING ---
print("Generating Polynomial Features...")
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly.fit(X_train_raw[:1000])
total_features = len(poly.get_feature_names_out())

scaler = StandardScaler()
scaler.fit(poly.transform(X_train_raw[:5000]))
gc.collect()

# Process Train
train_chunks = np.array_split(X_train_raw, 12)
np.save(local_train_poly, np.empty((len(X_train_raw), total_features), dtype=np.float16))
train_mmap = np.load(local_train_poly, mmap_mode='r+')
start = 0
for chunk in train_chunks:
    train_mmap[start:start+len(chunk)] = scaler.transform(poly.transform(chunk)).astype(np.float16)
    start += len(chunk)
    del chunk; gc.collect()
del train_mmap, train_chunks; gc.collect()

# Process Test
test_chunks = np.array_split(X_test_raw, 4)
np.save(local_test_poly, np.empty((len(X_test_raw), total_features), dtype=np.float16))
test_mmap = np.load(local_test_poly, mmap_mode='r+')
start = 0
for chunk in test_chunks:
    test_mmap[start:start+len(chunk)] = scaler.transform(poly.transform(chunk)).astype(np.float16)
    start += len(chunk)
    del chunk; gc.collect()
del test_mmap, test_chunks, X_train_raw, X_test_raw; gc.collect()

# --- 3. DCN TRAINING ---
print("Training DCN-V2...")
y_train_one_hot = to_categorical(y_train, num_classes=num_classes)
X_train_poly_mmap = np.load(local_train_poly, mmap_mode='r')

val_split_idx = int(len(X_train_poly_mmap) * 0.9)
train_ds = tf.data.Dataset.from_tensor_slices((X_train_poly_mmap[:val_split_idx], y_train_one_hot[:val_split_idx]))
train_ds = train_ds.shuffle(10000).batch(2048).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_train_poly_mmap[val_split_idx:], y_train_one_hot[val_split_idx:]))
val_ds = val_ds.batch(2048).prefetch(tf.data.AUTOTUNE)

del X_train_poly_mmap, y_train_one_hot; gc.collect()

# Model Architecture
inp = Input(shape=(total_features,))
x0 = inp
x_cross = x0
for _ in range(4):
    x_cross = Dense(total_features, activation='linear')(x_cross)
    x_cross = x0 * x_cross + x_cross
x_deep = Dense(512, activation='swish')(inp)
x_deep = BatchNormalization()(x_deep)
x_deep = Dropout(0.4)(x_deep)
x_deep = Dense(256, activation='swish')(x_deep)
x_deep = BatchNormalization()(x_deep)
x_deep = Dropout(0.3)(x_deep)
combined = Concatenate()([x_cross, x_deep])
x = Dense(128, activation='swish')(combined)
x = Dropout(0.2)(x)
out = Dense(num_classes, activation='softmax', dtype='float32')(x)

model = Model(inp, out)
model_filename = f'{output_dir}/polyquant_model.keras'
checkpoint = ModelCheckpoint(model_filename, monitor='val_accuracy', save_best_only=True, mode='max', verbose=0)

optimizer = Adam(learning_rate=0.0001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=10, mode='max', restore_best_weights=True, verbose=0)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=0)

print("Fitting model (this takes ~20-40 mins)...")
history = model.fit(train_ds, epochs=100, validation_data=val_ds, callbacks=[checkpoint, early_stop, reduce_lr], verbose=1)

# --- NEW: PLOT TRAINING CURVE ---
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss During Training')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True)
curve_path = f'{output_dir}/training_loss_curve.png'
plt.savefig(curve_path, dpi=300)
plt.close()
print(f"Training curve saved to {curve_path}")

del train_ds, val_ds; gc.collect()

model.load_weights(model_filename)
print("Generating DCN Predictions...")
X_test_poly_mmap = np.load(local_test_poly, mmap_mode='r')
nn_probs = model.predict(X_test_poly_mmap, batch_size=2048)
np.save(f'{output_dir}/nn_probs.npy', nn_probs)
del X_test_poly_mmap, model; gc.collect()

# --- 4. CATBOOST & BASELINE ---
print("Training CatBoost & Baseline...")
train_pool = Pool(data=X_train_raw_cb, label=y_train)
test_pool = Pool(data=X_test_raw_cb, label=y_test)

cat_model = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=10, loss_function='MultiClass', task_type='GPU', devices='0', verbose=False, early_stopping_rounds=50)
cat_model.fit(train_pool, eval_set=test_pool)
cat_probs = cat_model.predict_proba(test_pool)
np.save(f'{output_dir}/cat_probs.npy', cat_probs)

# RF Baseline
rf = RandomForestClassifier(n_estimators=50, max_depth=15, n_jobs=1, random_state=42)
rf.fit(X_train_raw_cb, y_train)
rf_preds = rf.predict(X_test_raw_cb)
rf_acc = accuracy_score(y_test, rf_preds)
print(f"Random Forest Baseline: {rf_acc:.4f}")

# --- 5. PLOTTING ---
print("Generating Plots...")
y_test = np.load(f'{output_dir}/y_test.npy')
nn_probs = np.load(f'{output_dir}/nn_probs.npy')
cat_probs = np.load(f'{output_dir}/cat_probs.npy')
class_names = np.load(f'{output_dir}/classes.npy', allow_pickle=True)

ensemble_probs = 0.4 * nn_probs + 0.6 * cat_probs
ensemble_preds = np.argmax(ensemble_probs, axis=1)
cat_preds = np.argmax(cat_probs, axis=1)

# Confusion Matrix
cm = confusion_matrix(y_test, ensemble_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
plt.figure(figsize=(20, 18))
sns.heatmap(cm_norm, annot=False, cmap='Blues', fmt='.2f', xticklabels=class_names, yticklabels=class_names)
plt.title('Normalized Confusion Matrix', fontsize=20)
plt.ylabel('True Label', fontsize=16)
plt.xlabel('Predicted Label', fontsize=16)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f'{output_dir}/confusion_matrix.png', dpi=300)
plt.close()

# F1 Comparison
cat_f1 = f1_score(y_test, cat_preds, average=None)
ens_f1 = f1_score(y_test, ensemble_preds, average=None)
f1_df = pd.DataFrame({'CatBoost': cat_f1, 'PolyQuant': ens_f1}, index=class_names)
plt.figure(figsize=(12, 10))
f1_df.plot(kind='barh', width=0.8, color=['#1f77b4', '#ff7f0e'])
plt.title('Per-Class F1-Score Comparison', fontsize=16)
plt.xlabel('F1-Score', fontsize=14)
plt.xlim(0.4, 1.0)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(f'{output_dir}/f1_comparison.png', dpi=300)
plt.close()

print("ALL DONE!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.5 MB/s eta 0:00:00
Copying dataset...
Loading dataset...
Preprocessing...
Generating Polynomial Features...
Training DCN-V2...
Fitting model (this takes ~20-40 mins)...
Epoch 1/100
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 41s 23ms/step - accuracy: 0.6884 - loss: 1.3410 - val_accuracy: 0.7569 - val_loss: 0.6983 - learning_rate: 1.0000e-04
Epoch 2/100
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - accuracy: 0.7561 - loss: 0.6649 - val_accuracy: 0.7789 - val_loss: 0.5620 - learning_rate: 1.0000e-04
Epoch 3/100
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - accuracy: 0.7753 - loss: 0.5675 - val_accuracy: 0.7849 - val_loss: 0.5260 - learning_rate: 1.0000e-04
Epoch 4/100
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.7843 - loss: 0.5337 - val_accuracy: 0.7899 - val_loss: 0.5104 - learning_rate: 1.0000e-04
Epoch 5/100
1196/1196 ━━━━━━━━━━━━━━━━━━━━ 21s 18ms/step - accuracy: 0.7900 - loss: 0.5111 - val_accuracy: 0.7930 - val_loss: 0.

Random Forest Baseline: 0.7951
Generating Plots...
ALL DONE!


<Figure size 1200x1000 with 0 Axes>